# 3장 — 도구를 드는 손: 도구 호출

웹 챕터(3장)와 **같은 질문 20개, 같은 수치**로 라우팅의 저울질을 계산해 봅니다.
임계값을 훑으며(sweep) 놓침/헛발 곡선을 그리고, 점수 구간별 교정 표를 만들어요.

> 🤖 **AI 튜터와 함께 공부하세요**: 아래 셀을 실행하면 화면 오른쪽 아래에 튜터 버튼이 뜹니다.
> 로컬 Ollama 가 꺼져 있어도 노트북 학습에는 지장이 없습니다.

In [ ]:
from tutor import chat
chat('ch3')  # 이 챕터를 아는 AI 튜터 (로컬 Ollama — 없어도 학습엔 지장 없음)

## 1. 오늘 들어온 질문 20개: 웹 실험실과 동일 데이터

각 질문에는 정답 종류(`calc` 계산 필요 / `search` 검색 필요 / `plain` 그냥 답)와,
라우터가 매긴 **계산 도구 신뢰도 점수**(0~1)가 붙어 있습니다.
점수가 **임계값 이상**이면 계산 도구를 호출합니다.

In [ ]:
# (질문, 종류, 점수) — 웹 ToolRouteLab.vue 와 완전 동일한 하드코딩 데이터
ITEMS = [
    ('1,247 × 38은 얼마지?', 'calc', 0.97),
    ('월 매출 3.2억이 12% 성장하면 내년은?', 'calc', 0.92),
    ('회의실 대여료 시간당 45,000원, 6.5시간이면?', 'calc', 0.88),
    ('환율 1,383원일 때 2,900달러는 원화로?', 'calc', 0.83),
    ('작년 1,240에서 올해 1,517로, 몇 % 늘었는지 계산해 줘', 'calc', 0.74),
    ('세 분기 성장률 4%, 7%, 5.5%의 평균 구해줘', 'calc', 0.61),
    ('이 숫자들 합계가 맞는지 봐줘', 'calc', 0.48),
    ('인원 23명이 3개 조로 나뉘면 조당 몇 명꼴?', 'calc', 0.42),
    ('다음 주 서울 날씨 어때?', 'search', 0.03),
    ('경쟁사 신제품 출시일이 언제였지?', 'search', 0.06),
    ('최신 환율이 얼마야?', 'search', 0.55),
    ('우리 회사 연차 규정 알려줘', 'search', 0.09),
    ('그 보고서 원본 링크 찾아줘', 'search', 0.05),
    ('업계 평균 이직률 통계 찾아봐', 'search', 0.52),
    ('회의 내용 요약 좀 해줘', 'plain', 0.04),
    ('이 문장 더 정중하게 다듬어줘', 'plain', 0.02),
    ('보고서 제목 아이디어 3개만', 'plain', 0.07),
    ('숫자 감각이라는 표현이 무슨 뜻이야?', 'plain', 0.31),
    ('발표 순서 추천해줘', 'plain', 0.08),
    ('퍼센트포인트가 뭐야?', 'plain', 0.38),
]
assert len(ITEMS) == 20
assert [sum(1 for _, k, _s in ITEMS if k == kk) for kk in ('calc', 'search', 'plain')] == [8, 6, 6]

def aggregate(threshold):
    """임계값 하나에 대한 (정답, 놓침, 헛발) 집계 — 호출 조건: 점수 >= 임계값."""
    tp = fn = fp = tn = 0
    for _t, k, s in ITEMS:
        need, call = (k == 'calc'), (s >= threshold)
        if need and call: tp += 1
        elif need and not call: fn += 1
        elif not need and call: fp += 1
        else: tn += 1
    return {'ok': tp + tn, 'miss': fn, 'fa': fp, 'tp': tp, 'tn': tn}

base = aggregate(0.5)
print('임계값 0.50 →', base)
assert base == {'ok': 16, 'miss': 2, 'fa': 2, 'tp': 6, 'tn': 10}  # 웹 본문 수치와 일치
print('✅ 웹 실험실(임계값 0.50: 16/20, 놓침 2, 헛발 2)과 일치')

## 2. 임계값 스윕: 놓침과 헛발의 저울

임계값을 0부터 1까지 훑으며 두 오류가 어떻게 맞바꿔지는지 그려 봅니다.
**완벽(20/20)은 없고**, 최고 정확도 18/20이 서로 다른 두 구간에서 나옵니다:
약 0.40(놓침 0·헛발 2)과 약 0.58(놓침 2·헛발 0). 같은 점수, 다른 오류.

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

ts = [i / 100 for i in range(0, 101)]
sweep = [aggregate(t) for t in ts]

best = max(a['ok'] for a in sweep)
assert best == 18  # 어떤 임계값도 20/20 은 불가능
assert aggregate(0.40) == {'ok': 18, 'miss': 0, 'fa': 2, 'tp': 8, 'tn': 10}
assert aggregate(0.58) == {'ok': 18, 'miss': 2, 'fa': 0, 'tp': 6, 'tn': 12}
print('최고 정확도:', best, '/ 20: 0.40(놓침0·헛발2)과 0.58(놓침2·헛발0) 두 얼굴')

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(ts, [a['ok'] for a in sweep], color='#10b981', lw=2, label='정답 수')
ax.plot(ts, [a['miss'] for a in sweep], color='#f43f5e', lw=2, label='놓침 (필요한데 안 씀)')
ax.plot(ts, [a['fa'] for a in sweep], color='#f59e0b', lw=2, label='헛발 (불필요한데 씀)')
for t0 in (0.40, 0.50, 0.58):
    ax.axvline(t0, ls=':', lw=1, color='#94a3b8')
    ax.annotate(f'{t0:.2f}', (t0, 19.3), ha='center', fontsize=9, color='#64748b')
ax.set_xlabel('임계값 (이 점수부터 도구 호출)')
ax.set_ylabel('개수 (질문 20개 중)')
ax.set_title('임계값 스윕: 놓침·헛발은 맞바꿔질 뿐, 함께 0이 되지 않는다')
ax.legend(loc='center right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. 점수를 확률처럼 읽어도 될까: 교정(calibration) 표

점수 구간별로 **실제 계산 문제의 비율**, 곧 조건부 확률 P(계산 문제 | 점수 구간)을 세어 봅니다.
점수와 실제 비율이 얼추 맞으면 "교정이 잘 됐다"고 하고, 그래야 점수를 확률처럼 믿고
임계값·재시도(4장)·검수(6장) 설계에 쓸 수 있어요.

In [ ]:
BINS = [(0.0, 0.2), (0.2, 0.4), (0.4, 0.6), (0.6, 0.8), (0.8, 1.01)]
table = []
for lo, hi in BINS:
    items = [(k, s) for _t, k, s in ITEMS if lo <= s < hi]
    pos = sum(1 for k, _s in items if k == 'calc')
    table.append((f'{lo:.1f}~{min(hi, 1.0):.1f}', pos, len(items)))
    pct = f'{pos / len(items) * 100:3.0f}%' if items else '  — '
    print(f'점수 {lo:.1f}~{min(hi, 1.0):.1f} : 실제 계산 문제 {pos}/{len(items)}  ({pct})')

assert [(p, n) for _l, p, n in table] == [(0, 8), (0, 2), (2, 4), (2, 2), (4, 4)]
print('\n✅ 웹 ④ 교정 표와 일치. 0.4~0.6 구간은 실제로 반반(50%), 0.8 이상은 100%')

## 4. 오늘의 발견

- 도구 호출은 **분류 결정**: 신뢰도 점수 → 임계값 → 호출/패스.
- 임계값은 **놓침 ↔ 헛발을 맞바꾸는 다이얼**. 완벽한 지점은 없고(최고 18/20),
  같은 정확도라도 오류의 종류가 다르다(0.40 vs 0.58). 선택 기준은 **업무의 오류 비용**.
- 점수를 확률처럼 읽으려면 **교정**(구간별 실제 비율)부터 검증. "점수는 주장, 교정은 검증".
- 도구 호출도 사슬(2장)의 한 걸음. 라우팅 정확도가 p_i 를 정한다.

다음 장(4장 다시 뽑기)에서 틀린 걸음을 되살리는 첫 무기, 재시도와 Best-of-N 을 계산합니다.
**웹 챕터로 돌아가 연습 10문제를 풀면 진도가 채워져요.**